# Startup Growth Analytics — Data Source Audit & Planning

**Where this fits:** before writing a single line of cleaning code, we need to know exactly
what data we already have, what's still missing, where each missing piece would come from,
and — honestly — whether it's actually obtainable for free at the country-year granularity
this project needs (2005–2025, ~200 countries).

This notebook does **no cleaning**. It's reconnaissance. By the end you should be able to
answer, for every block in the final dataset design:

1. Do we have it already? (World Bank)
2. Can we get it the *same way* we got World Bank data? (WGI)
3. Do we need a new connector, and does one realistically exist? (UNESCO, WIPO)
4. Is there a genuine free source, or do we need to make an explicit, documented compromise? (Startup data)

Module 2 (cleaning) only gets written once these questions have real answers — otherwise
we'd be designing a pipeline around data that doesn't exist.


In [1]:
import os
from pathlib import Path

# Walk up until we find the project root (marked by config/ and scripts/ both existing)
while not (Path("config").exists() and Path("scripts").exists()):
    os.chdir("..")
    if Path.cwd() == Path.cwd().parent:  # hit filesystem root, stop
        raise RuntimeError("Could not locate project root")

print("Working directory set to:", os.getcwd())


Working directory set to: c:\Users\AJAY\Desktop\startup_implementation


## 1. What Module 1 actually gave us

Module 1 pulled **26 World Bank indicators** via the WB API into `data/raw/world_bank/*.csv`.
Every file shares the same long-format schema:

`country_code | country_name | year | indicator_code | value`

Let's load it for real and see what we're working with, instead of assuming.

In [2]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw/world_bank")   # relative to notebooks/ inside the project
files = sorted(RAW_DIR.glob("*.csv"))
print(f"{len(files)} indicator files found\n")
for f in files[:5]:
    print(" -", f.name)
print(" ...")


35 indicator files found

 - BX.KLT.DINV.CD.WD.csv
 - EG.FEC.RNEW.ZS.csv
 - EN.GHG.CO2.PC.CE.AR5.csv
 - EN.POP.DNST.csv
 - FM.LBL.BMNY.GD.ZS.csv
 ...


In [3]:
sample = pd.read_csv(RAW_DIR / "NY.GDP.MKTP.CD.csv")
print(sample.shape)
sample.head()


(5379, 5)


,country_code,country_name,year,indicator_code,value
0,AFE,Africa Eastern and Southern,2025,NY.GDP.MKTP.CD,1.358694e+12
1,AFE,Africa Eastern and Southern,2024,NY.GDP.MKTP.CD,1.252564e+12
2,AFE,Africa Eastern and Southern,2023,NY.GDP.MKTP.CD,1.179122e+12
3,AFE,Africa Eastern and Southern,2022,NY.GDP.MKTP.CD,1.226461e+12
4,AFE,Africa Eastern and Southern,2021,NY.GDP.MKTP.CD,1.113060e+12


In [4]:
# How many distinct entities (countries + aggregates) does WB report per indicator?
entities = sample[["country_code", "country_name"]].drop_duplicates()
print("Distinct entities in one indicator file:", len(entities))
print("Year range:", sample.year.min(), "-", sample.year.max())


Distinct entities in one indicator file: 261
Year range: 2005 - 2025


### The aggregates problem (why Module 2 can't just `groupby` blindly)

World Bank mixes real sovereign countries with regional/income aggregates in the *same*
`country_code` column — `WLD` (World), `EAS` (East Asia & Pacific), `OECD`, `HIC` (High income),
`SSA` (Sub-Saharan Africa), etc. They are **not** flagged by any obvious pattern in the code
itself (they're still 3-letter codes), so we can't filter with a regex or a length check.

The only reliable way is to cross-reference against the World Bank's own **country metadata**
endpoint, which tags each entity with `region.value`. Real countries have a real region;
aggregates have `region.value == "Aggregates"`.

`GET https://api.worldbank.org/v2/country?format=json&per_page=400`

This sandbox can't reach `api.worldbank.org` (network is restricted here), so the cell below
is written to run **on your machine**, where Module 1 already successfully called this same
API. It's wrapped so it won't crash this notebook — it'll just tell you it needs to run locally.

In [5]:
import requests

def fetch_wb_country_metadata():
    url = "https://api.worldbank.org/v2/country?format=json&per_page=400"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    _, records = r.json()
    meta = pd.DataFrame(records)
    meta["region_value"] = meta["region"].apply(lambda x: x.get("value"))
    meta["is_aggregate"] = meta["region_value"].eq("Aggregates")
    return meta[["id", "name", "region_value", "is_aggregate"]]

try:
    wb_meta = fetch_wb_country_metadata()
    n_agg = wb_meta.is_aggregate.sum()
    n_countries = (~wb_meta.is_aggregate).sum()
    print(f"Real countries: {n_countries} | Aggregates: {n_agg}")
    wb_meta.to_csv("data/raw/world_bank_country_metadata.csv", index=False)
    print("Saved -> data/raw/world_bank_country_metadata.csv (Module 2 will read this to filter)")
except requests.exceptions.RequestException as e:
    print("Could not reach api.worldbank.org from this environment.")
    print("Run this cell on your own machine (same place Module 1 ran) — it will work there.")
    print(f"({e.__class__.__name__})")


Real countries: 217 | Aggregates: 78
Saved -> data/raw/world_bank_country_metadata.csv (Module 2 will read this to filter)


## 2. Governance data (WGI) — the good news

The project spec treats WGI as a separate external source needing its own connector
(`connectors/world_bank.py` was even scaffolded with that assumption). **It isn't.**

The Worldwide Governance Indicators are published *through the same World Bank indicators
API* Module 1 already uses — they just have their own indicator codes:

| Dimension | WB Indicator Code |
|---|---|
| Voice & Accountability | `VA.EST` |
| Political Stability | `PV.EST` |
| Government Effectiveness | `GE.EST` |
| Regulatory Quality | `RQ.EST` |
| Rule of Law | `RL.EST` |
| Control of Corruption | `CC.EST` |

**Decision: no new connector needed.** We just add these 6 codes to `config/indicators.csv`
and re-run Module 1's existing downloader. That single change closes the entire "Governance"
block of the final dataset design.

In [6]:
wgi_rows = [
    {"variable_name": "government_effectiveness", "display_name": "Government Effectiveness",
     "indicator_code": "GE.EST", "category": "Governance", "unit": "Index (-2.5 to 2.5)", "required": "Yes"},
    {"variable_name": "regulatory_quality", "display_name": "Regulatory Quality",
     "indicator_code": "RQ.EST", "category": "Governance", "unit": "Index (-2.5 to 2.5)", "required": "Yes"},
    {"variable_name": "rule_of_law", "display_name": "Rule of Law",
     "indicator_code": "RL.EST", "category": "Governance", "unit": "Index (-2.5 to 2.5)", "required": "Yes"},
    {"variable_name": "control_of_corruption", "display_name": "Control of Corruption",
     "indicator_code": "CC.EST", "category": "Governance", "unit": "Index (-2.5 to 2.5)", "required": "Yes"},
    {"variable_name": "political_stability", "display_name": "Political Stability",
     "indicator_code": "PV.EST", "category": "Governance", "unit": "Index (-2.5 to 2.5)", "required": "Yes"},
    {"variable_name": "voice_accountability", "display_name": "Voice and Accountability",
     "indicator_code": "VA.EST", "category": "Governance", "unit": "Index (-2.5 to 2.5)", "required": "Yes"},
]
wgi_df = pd.DataFrame(wgi_rows)
wgi_df


,variable_name,display_name,indicator_code,category,unit,required
0,government_effectiveness,Government Effectiveness,GE.EST,Governance,Index (-2.5 to 2.5),Yes
1,regulatory_quality,Regulatory Quality,RQ.EST,Governance,Index (-2.5 to 2.5),Yes
2,rule_of_law,Rule of Law,RL.EST,Governance,Index (-2.5 to 2.5),Yes
3,control_of_corruption,Control of Corruption,CC.EST,Governance,Index (-2.5 to 2.5),Yes
4,political_stability,Political Stability,PV.EST,Governance,Index (-2.5 to 2.5),Yes
5,voice_accountability,Voice and Accountability,VA.EST,Governance,Index (-2.5 to 2.5),Yes


## 3. Education & R&D depth (UNESCO UIS) — what's actually missing

Check `config/indicators.csv` first: World Bank already gives us `rd_expenditure`
(`GB.XPD.RSDV.GD.ZS`) and `tertiary_enrollment` (`SE.TER.ENRR`). So UNESCO isn't needed to
*duplicate* those — it's needed for indicators the World Bank simply doesn't publish, mainly:

- **Researchers (FTE) per million inhabitants** — direct measure of R&D human capital
- More granular R&D personnel breakdowns

**Access:** UNESCO's live SDMX API was deprecated in 2020. The current, working route is the
**Bulk Data Download Service (BDDS)** — plain CSV files zipped by theme, no API key, no
authentication. The "Science" bulk file covers R&D.

In [7]:
import zipfile, io

UNESCO_RD_BULK_URL = "https://uis.unesco.org/sites/default/files/documents/bdds/rd.zip"  # verify exact link on the BDDS page before running

def fetch_unesco_rd(url=UNESCO_RD_BULK_URL, out_dir="data/raw/unesco"):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(out_dir)
    print("Extracted UNESCO R&D bulk files to", out_dir)

try:
    fetch_unesco_rd()
except requests.exceptions.RequestException as e:
    print("Could not reach uis.unesco.org from this environment.")
    print("Grab the exact current link from https://databrowser.uis.unesco.org/resources/bulk")
    print("(the BDDS file names/URLs get versioned by release month, so confirm before running)")
    print(f"({e.__class__.__name__})")


BadZipFile: File is not a zip file

## 4. Innovation output (WIPO) — patents & trademarks

Nothing in the World Bank set covers `patent_applications` or `trademark_applications` —
these are genuinely only available from WIPO's **IP Statistics Data Center**.

**Reality check:** unlike World Bank/UNESCO, WIPO does not expose a simple public REST/bulk
endpoint for this. Their data center is an interactive query tool (pick indicator → pick
countries → pick years → export CSV). There is no documented free API key for programmatic
pulls of this specific dataset.

**Decision for this project:** treat WIPO as a **manual one-time export**, not an automated
connector — go to https://www3.wipo.int/ipstats, export "Patent applications by office" and
"Trademark applications by office" as CSV for all countries/2005–2025, and drop the file into
`data/raw/wipo/`. `connectors/wipo.py` becomes a *loader* for that static export, not a live
fetcher. This is a normal, defensible thing to state in your methodology section — plenty of
official sources don't offer APIs.

## 5. Startup ecosystem data — decision (updated after deeper research)

Initial pass found no free, complete, real country-year panel of startup *funding* (counts,
deal sizes, unicorns) covering 2005–2025. Digging further turned up a real fix instead of a
compromise:

| Source | Real? | Coverage | Role |
|---|---|---|---|
| **World Bank Entrepreneurship Database** (`IC.BUS.NDNS.ZS`, `IC.BUS.NREG`) | Yes — official, CC BY-4.0 | 2006–2024, ~130–150 countries | **Core quantitative backbone** |
| GEM (Global Entrepreneurship Monitor) TEA rate | Yes — survey-based | ~50–100 countries/year | Supplementary column, coverage documented |
| StartupBlink / Startup Genome (GSER) rank | Yes — annual report | Top ~40–100 countries, ~2017/2020+ | Supplementary `startup_ecosystem_rank`, coverage documented |
| Funding amounts / deal counts / historical unicorn series | **No free real panel exists** | — | Dropped — paid (Crunchbase Pro/Dealroom) or synthetic only |

**Decision: `new_business_density` and `new_businesses_registered` become the real quantitative
core of the "startup" block.** They measure newly-registered limited-liability firms per
capita — the standard academic proxy for entrepreneurial activity when VC-funding panels
aren't available — and critically they come from the **same World Bank API** Module 1 already
uses. No new connector needed. GEM and StartupBlink/GSER get added later as supplementary,
partial-coverage columns with their limitations documented; raw funding/deal/unicorn counts
are dropped from the quantitative panel rather than faked.

This keeps the entire dataset 100% real data — no simulated rows anywhere.

## 6. Where this leaves the pipeline

| Block | Status | Action |
|---|---|---|
| Economic, Demographics, Digital, Finance (WB) | Done | Module 2: clean as planned |
| Governance (WGI) | No new connector needed | Add 6 indicator codes to `config/indicators.csv`, re-run Module 1 |
| Education & Innovation — R&D expenditure, tertiary enrollment | Done (WB) | — |
| Education & Innovation — researchers per capita | Missing | UNESCO BDDS bulk CSV (connector needed) |
| Innovation — patents, trademarks | Missing | WIPO manual export (loader, not live connector) |
| **Startup — new business density/registrations** | **No new connector needed** | **Add 2 indicator codes to `config/indicators.csv`, re-run Module 1** |
| Startup — supplementary (GEM TEA rate, StartupBlink/GSER rank) | Missing, partial coverage | Later addition, separate loaders, documented limitations |
| Startup — funding/deals/unicorn panel | **Confirmed unavailable free & real** | Dropped from quantitative panel |

**Net result: 8 new indicator codes (6 WGI + 2 Entrepreneurship) close the Governance block
completely and give the Startup block a real quantitative backbone — all through Module 1's
existing World Bank downloader.** The next cell builds the updated `indicators.csv`.

**Next session:** re-run Module 1 with these 8 codes added (34 indicators total), then start
Module 2 (`world_bank_long.csv` → `world_bank_wide.csv`) on the enlarged raw set, with the
aggregate-filtering logic from Section 1 built in properly.

In [ ]:
entrepreneurship_rows = [
    {"variable_name": "new_business_density", "display_name": "New Business Density",
     "indicator_code": "IC.BUS.NDNS.ZS", "category": "Startup", "unit": "New registrations per 1,000 people 15-64", "required": "Yes"},
    {"variable_name": "new_businesses_registered", "display_name": "New Businesses Registered",
     "indicator_code": "IC.BUS.NREG", "category": "Startup", "unit": "Count", "required": "Yes"},
]
entrepreneurship_df = pd.DataFrame(entrepreneurship_rows)

# Build the updated indicators.csv: original 26 + 6 WGI + 2 Entrepreneurship = 34
original = pd.read_csv("config/indicators.csv")
updated_indicators = pd.concat([original, wgi_df, entrepreneurship_df], ignore_index=True)

Path("config").mkdir(exist_ok=True)
updated_indicators.to_csv("config/indicators.csv", index=False)
print(f"config/indicators.csv updated: {len(original)} -> {len(updated_indicators)} indicators")
updated_indicators.tail(8)


config/indicators.csv updated: 35 -> 43 indicators


,variable_name,display_name,indicator_code,category,unit,required
35,government_effectiveness,Government Effectiveness,GE.EST,Governance,Index (-2.5 to 2.5),Yes
36,regulatory_quality,Regulatory Quality,RQ.EST,Governance,Index (-2.5 to 2.5),Yes
37,rule_of_law,Rule of Law,RL.EST,Governance,Index (-2.5 to 2.5),Yes
38,control_of_corruption,Control of Corruption,CC.EST,Governance,Index (-2.5 to 2.5),Yes
39,political_stability,Political Stability,PV.EST,Governance,Index (-2.5 to 2.5),Yes
40,voice_accountability,Voice and Accountability,VA.EST,Governance,Index (-2.5 to 2.5),Yes
41,new_business_density,New Business Density,IC.BUS.NDNS.ZS,Startup,"New registrations per 1,000 people 15-64",Yes
42,new_businesses_registered,New Businesses Registered,IC.BUS.NREG,Startup,Count,Yes
